# Домашнее задание 3: базы данных и Docker. Часть 1

База: Chinook (MySQL 8.0). Здесь только часть 1 (запросы и транзакции), 6 баллов.
Ячейки нужно запускать по порядку сверху вниз.

Перед началом: запущен Docker Desktop (иконка кита в трее/строке меню должна быть активна).

In [4]:
%pip install mysql-connector-python requests

Note: you may need to restart the kernel to use updated packages.


## Часть 1

### Задание 1.1. Запуск MySQL в Docker

Сначала скачиваем дамп Chinook в папку `init`. Образ mysql сам выполнит все `.sql` из
`/docker-entrypoint-initdb.d/`, но **только при первом запуске**, когда volume пустой.

In [5]:
import os
import urllib.request
# моя локальная папка
os.makedirs("homework", exist_ok=True)

url = "https://github.com/lerocha/chinook-database/releases/download/v1.4.5/Chinook_MySql_AutoIncrementPKs.sql"
dump_path = os.path.abspath("homework/chinook.sql")

if not os.path.exists(dump_path):
    urllib.request.urlretrieve(url, dump_path)

print(dump_path)

/Users/veramonina/Downloads/homework/homework/chinook.sql


Запуск контейнера. Что делает каждый ключ:

- `-e MYSQL_ROOT_PASSWORD=root` задаёт пароль root через переменную окружения;
- `-p 3307:3306` пробрасывает порт 3306 контейнера на порт 3307 компьютера (3306 может быть занят локальным MySQL);
- `-v chinook_data:/var/lib/mysql` именованный volume, в нём лежат файлы базы, поэтому данные переживают удаление контейнера;
- второй `-v` кладёт дамп в `/docker-entrypoint-initdb.d/` для автоматической инициализации.

Если контейнер с таким именем уже создавался раньше, сначала выполни `!docker rm -f chinook-mysql` (volume при этом сохранится).

In [6]:
!docker run -d \
  --name chinook-mysql-hw3 \
  -e MYSQL_ROOT_PASSWORD=root \
  -p 3307:3306 \
  -v chinook_data:/var/lib/mysql \
  -v "{dump_path}:/docker-entrypoint-initdb.d/chinook.sql:ro" \
  mysql:8.0

fb84d59b3ec122188e811a049a1c20e5763b5b9dd7f243641d4a84bac3d0cb34


In [7]:
!docker volume ls

DRIVER    VOLUME NAME
local     chinook_data


In [8]:
!docker volume inspect chinook_data

[
    {
        "CreatedAt": "2026-10-18T21:32:35Z",
        "Driver": "local",
        "Labels": null,
        "Mountpoint": "/var/lib/docker/volumes/chinook_data/_data",
        "Name": "chinook_data",
        "Options": null,
        "Scope": "local"
    }
]


In [9]:
!docker ps

CONTAINER ID   IMAGE       COMMAND                  CREATED         STATUS         PORTS                                         NAMES
fb84d59b3ec1   mysql:8.0   "docker-entrypoint.s…"   7 seconds ago   Up 6 seconds   0.0.0.0:3307->3306/tcp, [::]:3307->3306/tcp   chinook-mysql-hw3


In [10]:
import mysql.connector

test_con = mysql.connector.connect(
    host="127.0.0.1", port=3307, user="root", password="root",
    database="Chinook_AutoIncrement")
    
test_con.close()
print("База готова")


База готова


### Подключение через Python

In [11]:
import mysql.connector

con = mysql.connector.connect(
    host="127.0.0.1",
    port=3307,
    user="root",
    password="root",
    database="Chinook_AutoIncrement"
)
print("Подключение получилось:", con.is_connected())


cur = con.cursor(dictionary=True, buffered=True)

Подключение получилось: True


### Небольшое EDA, чтобы понять, как устроена база данных

In [12]:
def run(query):
    # выполняем один запрос и печатаем все строки
    cur.execute(query)
    for row in cur.fetchall():
        print(row)

In [13]:
run("SHOW TABLES")

{'Tables_in_Chinook_AutoIncrement': 'Album'}
{'Tables_in_Chinook_AutoIncrement': 'Artist'}
{'Tables_in_Chinook_AutoIncrement': 'Customer'}
{'Tables_in_Chinook_AutoIncrement': 'Employee'}
{'Tables_in_Chinook_AutoIncrement': 'Genre'}
{'Tables_in_Chinook_AutoIncrement': 'Invoice'}
{'Tables_in_Chinook_AutoIncrement': 'InvoiceLine'}
{'Tables_in_Chinook_AutoIncrement': 'MediaType'}
{'Tables_in_Chinook_AutoIncrement': 'Playlist'}
{'Tables_in_Chinook_AutoIncrement': 'PlaylistTrack'}
{'Tables_in_Chinook_AutoIncrement': 'Track'}
{'Tables_in_Chinook_AutoIncrement': 'customer_feedback'}


In [14]:
run("DESCRIBE Track")

{'Field': 'TrackId', 'Type': 'int', 'Null': 'NO', 'Key': 'PRI', 'Default': None, 'Extra': 'auto_increment'}
{'Field': 'Name', 'Type': 'varchar(200)', 'Null': 'NO', 'Key': '', 'Default': None, 'Extra': ''}
{'Field': 'AlbumId', 'Type': 'int', 'Null': 'YES', 'Key': 'MUL', 'Default': None, 'Extra': ''}
{'Field': 'MediaTypeId', 'Type': 'int', 'Null': 'NO', 'Key': 'MUL', 'Default': None, 'Extra': ''}
{'Field': 'GenreId', 'Type': 'int', 'Null': 'YES', 'Key': 'MUL', 'Default': None, 'Extra': ''}
{'Field': 'Composer', 'Type': 'varchar(220)', 'Null': 'YES', 'Key': '', 'Default': None, 'Extra': ''}
{'Field': 'Milliseconds', 'Type': 'int', 'Null': 'NO', 'Key': '', 'Default': None, 'Extra': ''}
{'Field': 'Bytes', 'Type': 'int', 'Null': 'YES', 'Key': '', 'Default': None, 'Extra': ''}
{'Field': 'UnitPrice', 'Type': 'decimal(10,2)', 'Null': 'NO', 'Key': '', 'Default': None, 'Extra': ''}


In [15]:
run("""
SELECT 'Artist' AS tbl, COUNT(*) AS n FROM Artist
UNION ALL SELECT 'Album', COUNT(*) FROM Album
UNION ALL SELECT 'Track', COUNT(*) FROM Track
UNION ALL SELECT 'Genre', COUNT(*) FROM Genre
UNION ALL SELECT 'Customer', COUNT(*) FROM Customer
UNION ALL SELECT 'Invoice', COUNT(*) FROM Invoice
UNION ALL SELECT 'InvoiceLine', COUNT(*) FROM InvoiceLine
""")

{'tbl': 'Artist', 'n': 275}
{'tbl': 'Album', 'n': 347}
{'tbl': 'Track', 'n': 3503}
{'tbl': 'Genre', 'n': 25}
{'tbl': 'Customer', 'n': 59}
{'tbl': 'Invoice', 'n': 412}
{'tbl': 'InvoiceLine', 'n': 2240}


In [16]:
# пропуски в таблице Track 
run("""
SELECT COUNT(*) AS total_tracks,
       SUM(Composer IS NULL) AS no_composer,
       SUM(AlbumId IS NULL) AS no_album,
       SUM(GenreId IS NULL) AS no_genre
FROM Track
""")

{'total_tracks': 3503, 'no_composer': Decimal('977'), 'no_album': Decimal('0'), 'no_genre': Decimal('0')}


In [17]:
# повторяющиеся названия треков
run("""
SELECT Name, COUNT(*) AS cnt
FROM Track
GROUP BY Name
HAVING COUNT(*) > 1
ORDER BY cnt DESC
LIMIT 10
""")

{'Name': 'The Number Of The Beast', 'cnt': 5}
{'Name': 'The Trooper', 'cnt': 5}
{'Name': '2 Minutes To Midnight', 'cnt': 5}
{'Name': 'Iron Maiden', 'cnt': 5}
{'Name': 'Hallowed Be Thy Name', 'cnt': 5}
{'Name': 'Wrathchild', 'cnt': 5}
{'Name': 'Dazed and Confused', 'cnt': 4}
{'Name': 'Sanctuary', 'cnt': 4}
{'Name': 'Running Free', 'cnt': 4}
{'Name': 'The Evil That Men Do', 'cnt': 4}


In [20]:
# цена и длительность треков: минимум, максимум, среднее
run("""
SELECT MIN(UnitPrice) AS min_price, MAX(UnitPrice) AS max_price,
       ROUND(AVG(UnitPrice), 2) AS avg_price,
       MIN(Milliseconds) AS min_ms, MAX(Milliseconds) AS max_ms,
       ROUND(AVG(Milliseconds) / 60000, 2) AS avg_minutes
FROM Track
""")

{'min_price': Decimal('0.99'), 'max_price': Decimal('1.99'), 'avg_price': Decimal('1.05'), 'min_ms': 1071, 'max_ms': 5286953, 'avg_minutes': Decimal('6.56')}


In [ ]:
# число треков по жанрам
# понадобиться потом, для задания про жанры
run("""
SELECT g.Name AS genre, COUNT(*) AS tracks
FROM Track AS  t
JOIN Genre g ON g.GenreId = t.GenreId
GROUP BY g.GenreId, g.Name
ORDER BY tracks DESC
""")

{'genre': 'Rock', 'tracks': 1297}
{'genre': 'Latin', 'tracks': 579}
{'genre': 'Metal', 'tracks': 374}
{'genre': 'Alternative & Punk', 'tracks': 332}
{'genre': 'Jazz', 'tracks': 130}
{'genre': 'TV Shows', 'tracks': 93}
{'genre': 'Blues', 'tracks': 81}
{'genre': 'Classical', 'tracks': 74}
{'genre': 'Drama', 'tracks': 64}
{'genre': 'R&B/Soul', 'tracks': 61}
{'genre': 'Reggae', 'tracks': 58}
{'genre': 'Pop', 'tracks': 48}
{'genre': 'Soundtrack', 'tracks': 43}
{'genre': 'Alternative', 'tracks': 40}
{'genre': 'Hip Hop/Rap', 'tracks': 35}
{'genre': 'Electronica/Dance', 'tracks': 30}
{'genre': 'Heavy Metal', 'tracks': 28}
{'genre': 'World', 'tracks': 28}
{'genre': 'Sci Fi & Fantasy', 'tracks': 26}
{'genre': 'Easy Listening', 'tracks': 24}
{'genre': 'Comedy', 'tracks': 17}
{'genre': 'Bossa Nova', 'tracks': 15}
{'genre': 'Science Fiction', 'tracks': 13}
{'genre': 'Rock And Roll', 'tracks': 12}
{'genre': 'Opera', 'tracks': 1}


In [ ]:
# артисты с количеством альбомов, сначала те, у кого их больше
cur.execute("""
SELECT ar.Name, COUNT(al.AlbumId) AS albums
FROM Artist AS  ar
LEFT JOIN Album al ON al.ArtistId = ar.ArtistId
GROUP BY ar.ArtistId, ar.Name
ORDER BY albums DESC, ar.Name
""")
for row in cur.fetchall():
    print(row["Name"], "-", row["albums"])

Iron Maiden - 21
Led Zeppelin - 14
Deep Purple - 11
Metallica - 10
U2 - 10
Ozzy Osbourne - 6
Pearl Jam - 5
Faith No More - 4
Foo Fighters - 4
Lost - 4
Van Halen - 4
Various Artists - 4
Audioslave - 3
Berliner Philharmoniker & Herbert Von Karajan - 3
Eugene Ormandy - 3
Gilberto Gil - 3
Guns N' Roses - 3
Jamiroquai - 3
Miles Davis - 3
Os Paralamas Do Sucesso - 3
Queen - 3
R.E.M. - 3
Red Hot Chili Peppers - 3
Santana - 3
The Office - 3
The Rolling Stones - 3
AC/DC - 2
Accept - 2
Amy Winehouse - 2
Antônio Carlos Jobim - 2
Battlestar Galactica - 2
Black Label Society - 2
Black Sabbath - 2
Caetano Veloso - 2
Cássia Eller - 2
Chico Science & Nação Zumbi - 2
Cidade Negra - 2
Creedence Clearwater Revival - 2
Djavan - 2
English Concert & Trevor Pinnock - 2
Eric Clapton - 2
Green Day - 2
Kiss - 2
Legião Urbana - 2
Lulu Santos - 2
Michael Tilson Thomas & San Francisco Symphony - 2
Milton Nascimento - 2
Nirvana - 2
Skank - 2
Smashing Pumpkins - 2
Spyro Gyra - 2
The Black Crowes - 2
The Cult - 2
The

In [24]:
# клиенты по странам
run("""
SELECT Country, COUNT(*) AS customers
FROM Customer
GROUP BY Country
ORDER BY customers DESC
""")

{'Country': 'USA', 'customers': 13}
{'Country': 'Canada', 'customers': 8}
{'Country': 'Brazil', 'customers': 5}
{'Country': 'France', 'customers': 5}
{'Country': 'Germany', 'customers': 4}
{'Country': 'United Kingdom', 'customers': 3}
{'Country': 'India', 'customers': 2}
{'Country': 'Portugal', 'customers': 2}
{'Country': 'Czech Republic', 'customers': 2}
{'Country': 'Denmark', 'customers': 1}
{'Country': 'Belgium', 'customers': 1}
{'Country': 'Austria', 'customers': 1}
{'Country': 'Finland', 'customers': 1}
{'Country': 'Hungary', 'customers': 1}
{'Country': 'Ireland', 'customers': 1}
{'Country': 'Italy', 'customers': 1}
{'Country': 'Netherlands', 'customers': 1}
{'Country': 'Poland', 'customers': 1}
{'Country': 'Spain', 'customers': 1}
{'Country': 'Sweden', 'customers': 1}
{'Country': 'Australia', 'customers': 1}
{'Country': 'Argentina', 'customers': 1}
{'Country': 'Chile', 'customers': 1}
{'Country': 'Norway', 'customers': 1}


### 1. Треки заданного жанра

In [ ]:
def tracks_by_genre(genre):
    query = """
    SELECT t.TrackId, t.Name, al.Title AS Album, t.UnitPrice
    FROM Track t
    JOIN Genre AS g ON g.GenreId = t.GenreId
    JOIN Album  AS al ON al.AlbumId = t.AlbumId  
    WHERE g.Name = %s
    ORDER BY t.Name
    """
    cur.execute(query, (genre,))
    return cur.fetchall()

In [28]:
rock_songs = tracks_by_genre("Rock")
print("Треков в жанре Jazz:", len(rock_songs))
for row in rock_songs[:10]:
    print(row)

Треков в жанре Jazz: 1297
{'TrackId': 3027, 'Name': '"40"', 'Album': 'War', 'UnitPrice': Decimal('0.99')}
{'TrackId': 570, 'Name': '(Da Le) Yaleo', 'Album': 'Supernatural', 'UnitPrice': Decimal('0.99')}
{'TrackId': 3057, 'Name': '(Oh) Pretty Woman', 'Album': 'Diver Down', 'UnitPrice': Decimal('0.99')}
{'TrackId': 709, 'Name': '(Wish I Could) Hideaway', 'Album': 'Chronicle, Vol. 2', 'UnitPrice': Decimal('0.99')}
{'TrackId': 2190, 'Name': '1/2 Full', 'Album': 'Riot Act', 'UnitPrice': Decimal('0.99')}
{'TrackId': 2671, 'Name': '19th Nervous Breakdown', 'Album': 'Hot Rocks, 1964-1971 (Disc 1)', 'UnitPrice': Decimal('0.99')}
{'TrackId': 1404, 'Name': '2 A.M.', 'Album': 'The X Factor', 'UnitPrice': Decimal('0.99')}
{'TrackId': 1319, 'Name': '2 Minutes To Midnight', 'Album': 'Live At Donington 1992 (Disc 2)', 'UnitPrice': Decimal('0.99')}
{'TrackId': 1573, 'Name': '2,000 Man', 'Album': 'Unplugged [Live]', 'UnitPrice': Decimal('0.99')}
{'TrackId': 355, 'Name': '200 Years Old', 'Album': 'Bongo 

### 2. Альбомы артиста по имени

In [ ]:
def albums_by_artist(name):
    query = """
    SELECT al.AlbumId, al.Title
    FROM Album AS  al
    JOIN Artist AS ar ON ar.ArtistId = al.ArtistId
    WHERE ar.Name = %s
    ORDER BY al.Title
    """
    cur.execute(query, (name,))
    return cur.fetchall()

In [32]:
for row in albums_by_artist("Iron Maiden"):
    print(row)


{'AlbumId': 94, 'Title': 'A Matter of Life and Death'}
{'AlbumId': 95, 'Title': 'A Real Dead One'}
{'AlbumId': 96, 'Title': 'A Real Live One'}
{'AlbumId': 97, 'Title': 'Brave New World'}
{'AlbumId': 98, 'Title': 'Dance Of Death'}
{'AlbumId': 99, 'Title': 'Fear Of The Dark'}
{'AlbumId': 100, 'Title': 'Iron Maiden'}
{'AlbumId': 101, 'Title': 'Killers'}
{'AlbumId': 102, 'Title': 'Live After Death'}
{'AlbumId': 103, 'Title': 'Live At Donington 1992 (Disc 1)'}
{'AlbumId': 104, 'Title': 'Live At Donington 1992 (Disc 2)'}
{'AlbumId': 105, 'Title': 'No Prayer For The Dying'}
{'AlbumId': 106, 'Title': 'Piece Of Mind'}
{'AlbumId': 107, 'Title': 'Powerslave'}
{'AlbumId': 108, 'Title': 'Rock In Rio [CD1]'}
{'AlbumId': 109, 'Title': 'Rock In Rio [CD2]'}
{'AlbumId': 110, 'Title': 'Seventh Son of a Seventh Son'}
{'AlbumId': 111, 'Title': 'Somewhere in Time'}
{'AlbumId': 112, 'Title': 'The Number of The Beast'}
{'AlbumId': 113, 'Title': 'The X Factor'}
{'AlbumId': 114, 'Title': 'Virtual XI'}


### 3. Отчёт по клиенту

Делаю два запроса. В первом считаются счета и сумма (`LEFT JOIN`, чтобы клиент без покупок
тоже вернулся, с нулями). Во втором ищется жанр, из которого куплено больше всего треков.
Если у двух жанров поровну, берётся тот, что раньше по алфавиту.

In [ ]:
def customer_report(customer_id):
    # имя, фамилия, email, число счетов и общая сумма
    query = """
    SELECT c.FirstName, c.LastName, c.Email,
           COUNT(i.InvoiceId) AS invoices,
           COALESCE(SUM(i.Total), 0) AS total
    FROM Customer AS c
    LEFT JOIN Invoice AS i ON i.CustomerId = c.CustomerId
    WHERE c.CustomerId = %s
    GROUP BY c.CustomerId
    """
    cur.execute(query, (customer_id,))
    report = cur.fetchone()
    if report is None:
        return None

    # жанр с наибольшим числом купленных треков
    query = """
    SELECT g.Name AS genre, SUM(il.Quantity) AS bought
    FROM Invoice AS i
    JOIN InvoiceLine AS il ON il.InvoiceId = i.InvoiceId
    JOIN Track AS t ON t.TrackId = il.TrackId
    JOIN Genre AS  g ON g.GenreId = t.GenreId
    WHERE i.CustomerId = %s
    GROUP BY g.GenreId, g.Name
    ORDER BY bought DESC, g.Name
    LIMIT 1
    """
    cur.execute(query, (customer_id,))
    top = cur.fetchone()
    report["top_genre"] = top["genre"] if top else None
    return report


In [38]:
print(customer_report(5))
print('\n')

{'FirstName': 'František', 'LastName': 'Wichterlová', 'Email': 'frantisekw@jetbrains.com', 'invoices': 7, 'total': Decimal('40.62'), 'top_genre': 'Rock'}




In [39]:
print(customer_report(50))
print('\n')

{'FirstName': 'Enrique', 'LastName': 'Muñoz', 'Email': 'enrique_munoz@yahoo.es', 'invoices': 7, 'total': Decimal('37.62'), 'top_genre': 'Rock'}




In [40]:
print(customer_report(10000000))  # такого клиента не доолжно быть

None


### 4. Треки с подстрокой в названии



In [41]:
def tracks_by_substring(part):
    query = """
    SELECT TrackId, Name 
    FROM Track 
    WHERE Name LIKE %s ORDER BY Name 
    """
    cur.execute(query, ("%" + part + "%",))
    return cur.fetchall()

In [42]:
found = tracks_by_substring("love")
print("Найдено треков:", len(found))
for row in found[:5]:
    print(row)

Найдено треков: 114
{'TrackId': 3045, 'Name': "(I Can't Help) Falling In Love With You"}
{'TrackId': 3471, 'Name': '(There Is) No Greater Love (Teo Licks)'}
{'TrackId': 3065, 'Name': "Ain't Talkin' 'bout Love"}
{'TrackId': 3084, 'Name': "Ain't Talkin' 'Bout Love"}
{'TrackId': 1608, 'Name': 'All My Love'}


In [43]:
found = tracks_by_substring("smile")
print("Найдено треков:", len(found))
for row in found[:5]:
    print(row)

Найдено треков: 2
{'TrackId': 2043, 'Name': 'Olodum - Smile (Instrumental)'}
{'TrackId': 136, 'Name': 'Phoney Smile Fake Hellos'}


### 5. Создание таблицы `customer_feedback`

5.1 Создание таблицы

In [44]:
cur.execute("DROP TABLE IF EXISTS customer_feedback") # чтобы можно было запускать несколько раз 

In [45]:
cur.execute("""
CREATE TABLE customer_feedback (
    feedback_id INT AUTO_INCREMENT PRIMARY KEY,
    customer_id INT NOT NULL,
    track_id INT NOT NULL,
    rating INT NOT NULL CHECK (rating BETWEEN 1 AND 5),
    comment TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (customer_id) REFERENCES Customer(CustomerId),
    FOREIGN KEY (track_id) REFERENCES Track(TrackId)
)
""")
con.commit()
print("Таблица создана")

Таблица создана


5.2  Вставка 6 записей через `executemany()` в одной транзакции

In [ ]:
def get_track_ids(genre, n):
    cur.execute("""
        SELECT t.TrackId
        FROM Track AS  t JOIN Genre AS  g ON g.GenreId = t.GenreId
        WHERE g.Name = %s
        ORDER BY t.TrackId
        LIMIT %s
    """, (genre, n))
    return [row["TrackId"] for row in cur.fetchall()]

In [47]:
rock_ids = get_track_ids("Rock", 3)
jazz_ids = get_track_ids("Jazz", 3)
latin_ids = get_track_ids("Latin", 4)

print("Rock:", rock_ids, "Jazz:", jazz_ids, "Latin:", latin_ids)

Rock: [1, 2, 3] Jazz: [63, 64, 65] Latin: [205, 206, 207, 208]


In [48]:
insert_query = """
INSERT INTO customer_feedback (customer_id, track_id, rating, comment)
VALUES (%s, %s, %s, %s)
"""

In [49]:
feedback_data = [
    (1, rock_ids[0], 5, "Круто"),
    (2, rock_ids[1], 3, "Хорошая песня и альбом"),
    (3, rock_ids[2], 2, None),            
    (4, jazz_ids[0], 4, "Семейная песня"),
    (5, jazz_ids[1], 1, "Не понравилась, слишком скучная"),
    (1, jazz_ids[2], 3, "Среднее"),
    (2, latin_ids[0], 5, "Отлично"),
    (3, latin_ids[1], 4, "Не советую слушать"),
    (4, latin_ids[2], 2, "Не понравилось"),
    (5, latin_ids[3], 1, "Прикольно")
]

In [50]:
cur.executemany(insert_query, feedback_data)
con.commit()
print("Вставлено строк:", len(feedback_data))

Вставлено строк: 10


In [ ]:
def show_feedback():
    cur.execute("""
        SELECT f.feedback_id, f.customer_id, g.Name AS genre, f.rating, f.comment
        FROM customer_feedback AS f
        JOIN Track AS t ON t.TrackId = f.track_id
        JOIN Genre AS g ON g.GenreId = t.GenreId
        ORDER BY f.feedback_id
    """)
    for row in cur.fetchall():
        print(row)


show_feedback()

{'feedback_id': 1, 'customer_id': 1, 'genre': 'Rock', 'rating': 5, 'comment': 'Круто'}
{'feedback_id': 2, 'customer_id': 2, 'genre': 'Rock', 'rating': 3, 'comment': 'Хорошая песня и альбом'}
{'feedback_id': 3, 'customer_id': 3, 'genre': 'Rock', 'rating': 2, 'comment': None}
{'feedback_id': 4, 'customer_id': 4, 'genre': 'Jazz', 'rating': 4, 'comment': 'Семейная песня'}
{'feedback_id': 5, 'customer_id': 5, 'genre': 'Jazz', 'rating': 1, 'comment': 'Не понравилась, слишком скучная'}
{'feedback_id': 6, 'customer_id': 1, 'genre': 'Jazz', 'rating': 3, 'comment': 'Среднее'}
{'feedback_id': 7, 'customer_id': 2, 'genre': 'Latin', 'rating': 5, 'comment': 'Отлично'}
{'feedback_id': 8, 'customer_id': 3, 'genre': 'Latin', 'rating': 4, 'comment': 'Не советую слушать'}
{'feedback_id': 9, 'customer_id': 4, 'genre': 'Latin', 'rating': 2, 'comment': 'Не понравилось'}
{'feedback_id': 10, 'customer_id': 5, 'genre': 'Latin', 'rating': 1, 'comment': 'Прикольно'}


5.3 Пакетное обновление: оценки +1 (но не выше 5) для треков одного жанра**

In [52]:
print("До обновления:")
show_feedback()

До обновления:
{'feedback_id': 1, 'customer_id': 1, 'genre': 'Rock', 'rating': 5, 'comment': 'Круто'}
{'feedback_id': 2, 'customer_id': 2, 'genre': 'Rock', 'rating': 3, 'comment': 'Хорошая песня и альбом'}
{'feedback_id': 3, 'customer_id': 3, 'genre': 'Rock', 'rating': 2, 'comment': None}
{'feedback_id': 4, 'customer_id': 4, 'genre': 'Jazz', 'rating': 4, 'comment': 'Семейная песня'}
{'feedback_id': 5, 'customer_id': 5, 'genre': 'Jazz', 'rating': 1, 'comment': 'Не понравилась, слишком скучная'}
{'feedback_id': 6, 'customer_id': 1, 'genre': 'Jazz', 'rating': 3, 'comment': 'Среднее'}
{'feedback_id': 7, 'customer_id': 2, 'genre': 'Latin', 'rating': 5, 'comment': 'Отлично'}
{'feedback_id': 8, 'customer_id': 3, 'genre': 'Latin', 'rating': 4, 'comment': 'Не советую слушать'}
{'feedback_id': 9, 'customer_id': 4, 'genre': 'Latin', 'rating': 2, 'comment': 'Не понравилось'}
{'feedback_id': 10, 'customer_id': 5, 'genre': 'Latin', 'rating': 1, 'comment': 'Прикольно'}


In [ ]:
# `LEAST(rating + 1, 5)` не даёт оценке уйти за пять. 
update_query = """
UPDATE customer_feedback AS f
JOIN Track AS t ON t.TrackId = f.track_id
JOIN Genre AS g ON g.GenreId = t.GenreId
SET f.rating = LEAST(f.rating + 1, 5) 
WHERE g.Name = %s
"""

In [54]:
cur.execute(update_query, ("Rock",))
print("\nИзменено строк:", cur.rowcount)
con.commit()


print("\nПосле обновления:")
show_feedback()


Изменено строк: 2

После обновления:
{'feedback_id': 1, 'customer_id': 1, 'genre': 'Rock', 'rating': 5, 'comment': 'Круто'}
{'feedback_id': 2, 'customer_id': 2, 'genre': 'Rock', 'rating': 4, 'comment': 'Хорошая песня и альбом'}
{'feedback_id': 3, 'customer_id': 3, 'genre': 'Rock', 'rating': 3, 'comment': None}
{'feedback_id': 4, 'customer_id': 4, 'genre': 'Jazz', 'rating': 4, 'comment': 'Семейная песня'}
{'feedback_id': 5, 'customer_id': 5, 'genre': 'Jazz', 'rating': 1, 'comment': 'Не понравилась, слишком скучная'}
{'feedback_id': 6, 'customer_id': 1, 'genre': 'Jazz', 'rating': 3, 'comment': 'Среднее'}
{'feedback_id': 7, 'customer_id': 2, 'genre': 'Latin', 'rating': 5, 'comment': 'Отлично'}
{'feedback_id': 8, 'customer_id': 3, 'genre': 'Latin', 'rating': 4, 'comment': 'Не советую слушать'}
{'feedback_id': 9, 'customer_id': 4, 'genre': 'Latin', 'rating': 2, 'comment': 'Не понравилось'}
{'feedback_id': 10, 'customer_id': 5, 'genre': 'Latin', 'rating': 1, 'comment': 'Прикольно'}


5.4 Невалидные транзакции

В одной транзакции вставляю валидную запись, а потом запись с оценкой 7. Она нарушает `CHECK`,
MySQL выдаёт ошибку, и мы делаем `rollback()`. Валидная запись, которая шла первой, тоже должна пропасть.

In [55]:
cur.execute("SELECT COUNT(*) AS n FROM customer_feedback")
print("Записей до транзакции:", cur.fetchone()["n"])

Записей до транзакции: 10


In [56]:
# 1) валидная запись
cur.execute(insert_query, (1, rock_ids[0], 4, "валидная запись"))
print("Валидная запись вставлена (пока без commit)")

Валидная запись вставлена (пока без commit)


In [57]:
# сколько записей было до транзакции
cur.execute("SELECT COUNT(*) AS n FROM customer_feedback")
print("Записей до транзакции:", cur.fetchone()["n"])

try:
    # 1) валидная запись
    cur.execute(insert_query, (1, rock_ids[0], 4, "валидная запись"))
    print("Валидная запись вставлена (пока без commit)")

    # 2) невалидная запись: рейтинг 7 нарушает CHECK
    cur.execute(insert_query, (1, rock_ids[0], 7, "невалидная запись"))

    con.commit()
    print("commit выполнен")
except mysql.connector.Error as err:
    # любая ошибка базы попадает сюда, и мы отменяем всю транзакцию
    print("Ошибка:", err)
    con.rollback()
    print("Выполнен rollback")

# сколько записей стало после транзакции
cur.execute("SELECT COUNT(*) AS n FROM customer_feedback")
print("Записей после транзакции:", cur.fetchone()["n"])

# есть ли в таблице тестовые записи
cur.execute("SELECT * FROM customer_feedback WHERE comment LIKE %s", ("ТЕСТ ОТКАТА%",))
print("Найдено тестовых записей:", len(cur.fetchall()))

Записей до транзакции: 11
Валидная запись вставлена (пока без commit)
Ошибка: 3819 (HY000): Check constraint 'customer_feedback_chk_1' is violated.
Выполнен rollback
Записей после транзакции: 10
Найдено тестовых записей: 0


Количество записей не изменилось и тестовых записей в таблице нет: валидная вставка откатилась
вместе с невалидной, то есть транзакция сработала как единое целое.

In [58]:
cur.close()
con.close()